Accessing IPython cluster clients and printing their ids to check.

In [1]:
import ipyparallel as parallel

clients = parallel.Client(cluster_id='mpi')
clients.block = True  # use synchronous computations
print (clients.ids)


[0, 1, 2]


Importing mpi4py and numpy.

In [2]:
%%px
from mpi4py import MPI
import numpy as np

Implementing Convolution, you don't need to modify this code.

In [3]:
%%px
def convolve_func(main,kernel,KERNEL_DIM,DIMx,DIMy,upper_pad,lower_pad):
	num_pads = int((KERNEL_DIM - 1) / 2)
	conv = np.zeros(main.shape,dtype=int)
	main = np.concatenate((upper_pad,main,lower_pad))
	for i in range(DIMy):
		for j in range(DIMx):
			for k in range(KERNEL_DIM):
				for l in range(KERNEL_DIM):
					if j+l <= DIMx+1 and i+k>=num_pads and i+k<=DIMy:
						conv[j*DIMy+i] += main[(j+l)*DIMy+i-num_pads+k]#*kernel[k][l]
						#print(f"j={j}, l={l}, DIMy={DIMy}, i={i}, num_pads={num_pads}, k={k}, index={(j+l)*DIMy+i-num_pads+k}")
						#print(main)
	return conv

5 points:
Load MPI communicator, get the total number of processes and rank of the process                              

In [4]:
%%px
#and also print total number of processes and rank from each process
comm = MPI.COMM_WORLD
size = comm.Get_size()
rank = comm.Get_rank()

print(f"Processes: {size}, Rank: {rank}")


[stdout:1] Processes: 3, Rank: 1


[stdout:0] Processes: 3, Rank: 0


[stdout:2] Processes: 3, Rank: 2


5 points: 
Load or initialize data array and kernel array only in process 0(rank 0)                                      

In [5]:
%%px
DIMx = 0
DIMy = 0
KERNEL_DIM = 0
kernel = None
img = None

#Add a condition such that these intializations below should happen in only process 0
if rank == 0:
    img = np.array([[3, 9, 5, 9],[1, 7, 4, 3],[2, 1, 6, 5],[3, 9, 5, 9],[1, 7, 4, 3],[2, 1, 6, 5],[3, 9, 5, 9],[1, 7, 4, 3],[2, 1, 6, 5]], dtype='i')
    kernel = np.array([[0, 1, 0],[0, 0, 0],[0, -1, 0]], dtype='i')
    DIMx = img.shape[0]
    DIMy = img.shape[1]
    KERNEL_DIM = int(kernel.shape[0])

10 points: 
Broadcast data and kernel array sizes from process 0 to  all other processes                                 

In [6]:
%%px
#broadcast data and kernel array sizes (think why we are broadcasting sizes)
DIMx = comm.bcast(DIMx, root=0)
DIMy = comm.bcast(DIMy, root=0)
KERNEL_DIM = comm.bcast(KERNEL_DIM, root=0)

Initialize empty kernel array for all  processes except rank = 0, why we are not initialzing kernel array for rank 0?

Ans: Rank 0 is the process that contains the data we wish to broadcast. In order to receive this data, the other (not rank 0) processes must initialize appropriately sized arrays. However, initializing a kernel array for the rank 0 process would overwrite the kernel array, destroying the data we wish to broadcast. 

In [7]:
%%px
#initialize empty kernel array except for process 0(rank=0)
if rank != 0:
    kernel = np.empty((KERNEL_DIM, KERNEL_DIM), dtype='i')

10 points: 
Broadcast Kernel array from rank 0 to all other processes.                                                   

In [8]:
%%px
#broadcast kernel array from rank 0 to all other processes

#print(kernel.size, kernel.shape, kernel.dtype, rank, type(kernel))
comm.Bcast(kernel, root=0)

25 points: 
Split the rows in data array equally and scatter them from process 0 to all other process. To split them 
equally, number of rows in the data array must be an integer multiple of number of processes. MPI has ways 
to send unequal chunks of data between processses. But for here you can do with equal number.

In [9]:
%%px
#split and send data array to corresponding processses (you need to initialize a buffer to receive data from 
#process 0, similar to the random initializing done for kernel array)
rows_per_process = DIMx // size
buffer = np.empty((rows_per_process, DIMy), dtype='i')

comm.Scatter([img, rows_per_process * DIMy, MPI.INT], [buffer, rows_per_process * DIMy, MPI.INT], root=0)

#Here do we initialize buffer for process 0 also, if so why?(Hint: because of the function we are using to send 
#and receieve data)
''' The answer is that Scatter() operates on all processes. There isn't a "sender" process and other "receiving" processes as with broadcast,
with scatter all processes are receiver, and thus all processes must provide a valid receiving buffer. '''


Out[0:8]: ' The answer is that Scatter() operates on all processes. There isn\'t a "sender" process and other "receiving" processes as with broadcast,\nwith scatter all processes are receiver, and thus all processes must provide a valid receiving buffer. '

Out[1:8]: ' The answer is that Scatter() operates on all processes. There isn\'t a "sender" process and other "receiving" processes as with broadcast,\nwith scatter all processes are receiver, and thus all processes must provide a valid receiving buffer. '

Out[2:8]: ' The answer is that Scatter() operates on all processes. There isn\'t a "sender" process and other "receiving" processes as with broadcast,\nwith scatter all processes are receiver, and thus all processes must provide a valid receiving buffer. '

25 points: 
For convolution of kernel array and data array, you have to pass the kernel padding rows from one
process to another. please see objective for more details. Send and Recieve rows from one process 
to other. Careful with the data size and tags you are sending and receiving should match otherwise
commincator will wait for them indefintely.                                                                  

In [10]:
%%px
#send padding rows from one process to other (carefully observe which process to send data to which process and
# which process receives the data)
top = np.empty((1, DIMy), dtype='i')
bot = np.empty((1, DIMy), dtype='i')

if rank < (size - 1): comm.Send(buffer[-1, :], dest=(rank+1), tag=1)
if rank > 0: comm.Recv(top, source=(rank-1), tag=1)

if rank > 0: comm.Send(buffer[0, :], dest=(rank-1), tag=2)
if rank < (size - 1): comm.Recv(bot, source=(rank+1), tag=2)


# if rank == 0: pad_img = np.vstack([np.zeros((1, DIMy)), buffer, bot])
# elif rank == (size - 1): pad_img = np.vstack(top, buffer, [np.zeros((1, DIMy))])
# else: pad_img = np.vstack(top, buffer, bot)

Why we are loading data into process 0 and broadcasting input data to all other processes? are there any other methods to load data into all processes (not for evaluation)

Ans: Every process has its own memory space, which means that each process needs its own copy of the data. In order to avoid having each process load the data from disk, we instead have process 0 (the 'root' process) load the data once, then broadcast that data to all the other processes. This ensures that all processes have a copy of the data without each individually having to load the data from disk.

There are some alternative methods. One such method could entail having each process independently load the data from disk, though this runs the risk of conflicting I/O calls. Another method could have each process load only its specific chunk (set of rows) of data, but this requires the data to be split beforehand or at least to be easy to split.  

5 points: 
Perform Convolution operation by calling convolve_func() provided for each of the process with 
corresponding rows as arguments.                                                                             

In [11]:
%%px
#convolution function arguments
#main - data array (flattened array), only the part of the data array that is processed for each process
#kernel - kernel array
#DIMy - ColumnSize
#Dimx - RowSize
#upper_pad = upper padding row
#lower_pad = lower padding row

buffer = buffer.astype('i').flatten()
kernel = kernel.astype('i')
top = top.astype('i').flatten()
bot = bot.astype('i').flatten()

if rank == 0: conv = convolve_func(buffer, kernel, KERNEL_DIM, rows_per_process, DIMy, np.zeros(DIMy, dtype='i'), bot)
elif rank == (size - 1): conv = convolve_func(buffer, kernel, KERNEL_DIM, rows_per_process, DIMy, top, np.zeros(DIMy, dtype='i'))
else: conv = convolve_func(buffer, kernel, KERNEL_DIM, rows_per_process, DIMy, top, bot)

10 points: 
Gather the computed convolutional matrix rows to process 0.                                                 

In [12]:
%%px
#To receive data from all processes, process 0 should have a buffer

conv = conv.astype('i').flatten()
#print(conv.size)
convbuf = None
if rank == 0: 
    convbuf = np.empty(conv.size * size, dtype='i')
    #print(convbuf.dtype, conv.dtype)
comm.Gather(conv, convbuf, root=0)

[stdout:1] 12


[stdout:2] 12


[stdout:0] 12
int32 int32


Reshape the flattened array to match input dimensions

In [26]:
%%px
#Reshape the collected array to the input image dimensions
if rank == 0: 
    conv_img = convbuf.reshape(DIMy, DIMx)
    print(conv_img)

[stdout:0] [[20 29 37 21 23 38 49 32 23]
 [38 49 32 23 38 49 32 23 38]
 [49 32 23 38 49 32 23 38 49]
 [32 23 38 49 32 11 21 26 18]]


5 points: 
Test to check sequential convolution and MPI based parallel convolution outputs                               

In [27]:
%%px
if rank == 0:
    #main_grid is the actual input input image array that is flattened
    #convolution function arguments
    #main_grid - data array (flattened array)
    #kernel - kernel array
    #DIMy - ColumnSize
    #Dimx - RowSize
    #upper_pad = upper padding row
    #lower_pad = lower padding row
    
    #rename the below arguments according to your variable names
	main_grid = img.flatten()
	upper_pad = np.empty(DIMy, dtype=int)
	lower_pad = np.empty(DIMy, dtype=int)
    
    #Entire convolution in a single process
	conv1 = convolve_func(main_grid,kernel,KERNEL_DIM,DIMx,DIMy,upper_pad,upper_pad)
	conv1 = np.reshape(conv1, (-1, DIMx))
	print(conv1)
    #recvbuf is the convolution computed by parallel processes and gathered in process 0, 
    #if you named it different, modify that name below
    
    #Checking with parallel convolution output
	print(np.array_equal(conv1,conv_img))

[stdout:0] [[20 29 37 21 23 38 49 32 23]
 [38 49 32 23 38 49 32 23 38]
 [49 32 23 38 49 32 23 38 49]
 [32 23 38 49 32 11 21 26 18]]
True
